# Desafio DIO — YOLO: Transfer Learning + Evoluções 2D/3D/4D e Automação

Este notebook roda 100% no Google Colab (opção sugerida no próprio enunciado para quem não tem GPU local).

Etapas:
1. Instalação
2. Dataset (COCO subset + 2 classes novas: `capacete`, `extintor`)
3. Transfer learning com YOLOv8
4. Inferência (Figura 1 do enunciado)
5. Evolução 1: visão 2D->3D->4D
6. Evolução 2: automação de segurança orientada a eventos

In [ ]:
!pip install -q ultralytics filterpy
from ultralytics import YOLO
import torch, cv2, numpy as np
print('GPU disponível:', torch.cuda.is_available())

## 2. Dataset
Baixe seu `data.yaml` (gerado pelo `scripts/labelme_to_yolo.py`) ou use o `coco8` de teste rápido.

In [ ]:
from ultralytics.utils.downloads import download
download(['https://ultralytics.com/assets/coco8.zip'])
DATA_YAML = 'coco8/data.yaml'  # substitua pelo seu dataset combinado (COCO subset + capacete + extintor)

## 3. Transfer learning
Parte do `yolov8n.pt` (já treinado nas 80 classes do COCO) e retreina incluindo as classes novas,
congelando o backbone (`freeze=10`) para preservar o conhecimento pré-existente.

In [ ]:
model = YOLO('yolov8n.pt')
model.train(data=DATA_YAML, epochs=50, imgsz=640, freeze=10, project='runs/train', name='desafio_yolo')

## 4. Inferência — reproduz a Figura 1 do enunciado

In [ ]:
best = 'runs/train/desafio_yolo/weights/best.pt'
model = YOLO(best)
results = model.predict('coco8/images/train/000000000009.jpg', conf=0.3)
im_bgr = results[0].plot()
cv2.imwrite('outputs/deteccao_resultado.jpg', im_bgr)
from IPython.display import Image
Image('outputs/deteccao_resultado.jpg')

## 5. Evolução 1 — Visão 2D -> 3D -> 4D
Carrega o adaptador que adiciona profundidade (MiDaS) e rastreamento temporal (Kalman).

In [ ]:
import sys; sys.path.append('../advanced')
from vision_2d_3d_4d_adapter import Vision2D3D4DAdapter

adapter = Vision2D3D4DAdapter(focal_length_px=800)
frame = cv2.imread('coco8/images/train/000000000009.jpg')
result = model.predict(frame, verbose=False)[0]
tracked = adapter.process_frame(frame, result)
for obj in tracked:
    print(obj.class_name, 'pos3D=', obj.position_3d, 'vel3D=', obj.velocity_3d, 'pred_prox_pos=', obj.predicted_position_3d)

## 6. Evolução 2 — Automação de segurança orientada a eventos
Converte as detecções em eventos/alertas automáticos (ex.: pessoa sem capacete).

In [ ]:
sys.path.append('../advanced')
from automation_safety_system import build_default_safety_system, Detection

safety_system = build_default_safety_system()
detections = [
    Detection(class_name=model.names[int(b.cls[0])], confidence=float(b.conf[0]), bbox=tuple(b.xyxy[0].tolist()))
    for b in result.boxes
]

# simula 6 frames seguidos com a mesma condição para passar do debounce (min_consecutive_frames)
for _ in range(6):
    events = safety_system.process(detections)
for ev in events:
    print(ev.to_dict())